# Phase 1: Data Cleaning and Quality Validation
### IndiaKart E-Commerce Data Analytics  — Internmo Data Analyst Internship
**Analyst:** Kajal Rajak

## Step 1: Dataset Loading & Overview Profile

In [8]:
import os
import pandas as pd
import numpy as np
import zipfile

In [9]:
zip_path = "IndiaKart_data.zip"
extract_dir = "indiakart_data"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_dir)
print(f"Extracted '{zip_path}' into './{extract_dir}/'\n")

csv_files = [f for f in os.listdir(extract_dir) if f.endswith(".csv")]

raw_data = {}
copied_data_dict = {}

for file in sorted(csv_files):
    table_name = file.replace(".csv", "")
    df = pd.read_csv(os.path.join(extract_dir, file))

    raw_data[table_name] = df
    copied_data_dict[table_name] = df.copy()

print(f"✓ Loaded {len(copied_data_dict)} tables into working memory (`copied_data_dict`).")

Extracted 'IndiaKart_data.zip' into './indiakart_data/'

✓ Loaded 8 tables into working memory (`copied_data_dict`).


## Data Profiling

In [10]:
profile_summary = []

for table_name, df in copied_data_dict.items():
    profile_summary.append({
        'Table Name': f"{table_name}.csv",
        'Rows': len(df),
        'Columns': len(df.columns),
        'Numeric Columns': df.select_dtypes(include=np.number).shape[1],
        'Object Columns': df.select_dtypes(include='object').shape[1],
        'Total Missing Values': df.isnull().sum().sum(),
        'Memory Usage (MB)': round(df.memory_usage(deep=True).sum() / (1024 * 1024), 2)
    })

profile_df = pd.DataFrame(profile_summary)
print("=== Dataset Overview Profile ===")
display(profile_df)

=== Dataset Overview Profile ===


,Table Name,Rows,Columns,Numeric Columns,Object Columns,Total Missing Values,Memory Usage (MB)
0,customers DA.csv,10000,17,7,10,0,6.11
1,inventory DA.csv,1000,11,6,5,0,0.32
2,order_items DA.csv,100000,11,6,5,0,33.80
3,orders DA.csv,50000,19,7,12,17501,35.39
4,payments DA.csv,50000,13,3,10,50000,28.75
5,products DA.csv,1000,17,9,8,0,0.53
6,returns DA.csv,10000,10,1,9,0,5.45
7,suppliers DA.csv,200,14,5,9,0,0.11


## Step 2: Missing Values & Null Percentage Check

In [11]:
print("===Complete Column Data Types (dtypes) Audit across ALL Tables ===")

for table_name, df in copied_data_dict.items():
    print(f"\n--- Table: {table_name}.csv ({len(df)} rows, {len(df.columns)} columns) ---")
    null_count = df.isnull().sum()
    
    dtype_df = pd.DataFrame({
        'Column Name': df.columns,
        'Data Type': df.dtypes.values,
        'Null Count': null_count,
        "Rows Affected %": round((null_count / len(df)) * 100, 2)
    })
    
    display(dtype_df)

===Complete Column Data Types (dtypes) Audit across ALL Tables ===

--- Table: customers DA.csv (10000 rows, 17 columns) ---


,Column Name,Data Type,Null Count,Rows Affected %
customer_id,customer_id,object,0,0.0
first_name,first_name,object,0,0.0
last_name,last_name,object,0,0.0
email,email,object,0,0.0
phone,phone,int64,0,0.0
city,city,object,0,0.0
state,state,object,0,0.0
pincode,pincode,int64,0,0.0
gender,gender,object,0,0.0
age,age,int64,0,0.0



--- Table: inventory DA.csv (1000 rows, 11 columns) ---


,Column Name,Data Type,Null Count,Rows Affected %
inventory_id,inventory_id,object,0,0.0
product_id,product_id,object,0,0.0
warehouse_location,warehouse_location,object,0,0.0
quantity_available,quantity_available,int64,0,0.0
quantity_reserved,quantity_reserved,int64,0,0.0
reorder_level,reorder_level,int64,0,0.0
reorder_quantity,reorder_quantity,int64,0,0.0
last_restocked_date,last_restocked_date,object,0,0.0
unit_cost,unit_cost,int64,0,0.0
total_inventory_value,total_inventory_value,int64,0,0.0



--- Table: order_items DA.csv (100000 rows, 11 columns) ---


,Column Name,Data Type,Null Count,Rows Affected %
item_id,item_id,object,0,0.0
order_id,order_id,object,0,0.0
product_id,product_id,object,0,0.0
product_name,product_name,object,0,0.0
category,category,object,0,0.0
quantity,quantity,int64,0,0.0
unit_price,unit_price,int64,0,0.0
gst_rate,gst_rate,int64,0,0.0
gst_amount,gst_amount,float64,0,0.0
discount_amount,discount_amount,float64,0,0.0


... (6 outputs left out)


--- Table: returns DA.csv (10000 rows, 10 columns) ---


,Column Name,Data Type,Null Count,Rows Affected %
return_id,return_id,object,0,0.0
order_id,order_id,object,0,0.0
customer_id,customer_id,object,0,0.0
return_date,return_date,object,0,0.0
reason,reason,object,0,0.0
return_amount,return_amount,float64,0,0.0
refund_status,refund_status,object,0,0.0
refund_date,refund_date,object,0,0.0
return_condition,return_condition,object,0,0.0
remarks,remarks,object,0,0.0



--- Table: suppliers DA.csv (200 rows, 14 columns) ---


,Column Name,Data Type,Null Count,Rows Affected %
supplier_id,supplier_id,object,0,0.0
supplier_name,supplier_name,object,0,0.0
contact_person,contact_person,object,0,0.0
email,email,object,0,0.0
phone,phone,int64,0,0.0
city,city,object,0,0.0
state,state,object,0,0.0
pincode,pincode,int64,0,0.0
category,category,object,0,0.0
gstin,gstin,object,0,0.0


**Observation:** All amount-related columns are already stored in numeric data types (`int64` or `float64`). Therefore, no data type conversion is required for these columns. **17501** (35%) dates are missing in **delivered_date** and 100% missing dates found in **refund_date** column, which needs to be inspect.

## Step 3: Duplicate Rows Check

In [12]:
dedup_config = {
    "customers DA": ("customer_id"),
    "inventory DA": ("inventory_id"),
    "order_items DA": ("item_id"),
    "orders DA": ("order_id"),
    "payments DA": ("payment_id"),
    "products DA": ("product_id"),
    "returns DA": ("return_id"),
    "suppliers DA": ("supplier_id")
}

dedup_results = []

for table_name, pk in dedup_config.items():
    df = copied_data_dict[table_name].copy()
    duplicate_count = df[pk].duplicated(keep="first").sum()

    dedup_results.append({
        "Table": table_name,
        "Column": pk,
        "Detected": duplicate_count,
        "Detected %": round((duplicate_count / len(df)) * 100, 2)
    })

dedup_results_df = pd.DataFrame(dedup_results) 
print("---Duplicates Detection---")
display(dedup_results_df)

---Duplicates Detection---


,Table,Column,Detected,Detected %
0,customers DA,customer_id,0,0.0
1,inventory DA,inventory_id,0,0.0
2,order_items DA,item_id,0,0.0
3,orders DA,order_id,0,0.0
4,payments DA,payment_id,0,0.0
5,products DA,product_id,0,0.0
6,returns DA,return_id,0,0.0
7,suppliers DA,supplier_id,0,0.0


**Observation:** No duplicate ID'd detected in the Dataset.

## Step 4: Data Type Validation & Date Consistency Check

In [13]:
date_columns = {}

for table, df in copied_data_dict.items():
    detected_cols = [col for col in df.columns if "date" in col.lower()]

    if detected_cols:
        date_columns[table] = detected_cols

converted_date_dict = {}

for table, columns in date_columns.items():
    converted_date_dict[table] = {}

    for column in columns:
        converted_date_dict[table][column] = pd.to_datetime(copied_data_dict[table][column].astype("string").str.strip(), errors="coerce", dayfirst=True)

**Note:** Data type of all the date columns converted to **datetime**.

## Step 5: Outlier Analysis (`final_amount` > ₹5 Lakhs)

In [14]:
orders = copied_data_dict["orders DA"]

outlier_orders = orders[orders["final_amount"] > 500000].copy()
outlier_orders["outlier_flag"] = "Above ₹5 lakh"

print("Number of orders above ₹5 lakh:", len(outlier_orders))
display(outlier_orders)

Number of orders above ₹5 lakh: 312


,order_id,customer_id,order_date,order_time,status,city,state,pincode,total_amount,gst_amount,shipping_charge,discount_amount,final_amount,payment_method,shipping_partner,tracking_id,delivered_date,is_cod,channel,outlier_flag
33,ORD000034,CUST01867,10-03-2024,07:18:00,Delivered,Gwalior,Madhya Pradesh,714121,527178.05,87982.56,0,85216.51,527178.05,Credit Card,XpressBees,TRK426129266,17-03-2024,0,App,Above ₹5 lakh
34,ORD000035,CUST01370,26-05-2025,17:56:00,Cancelled,Mumbai,Maharashtra,135015,532061.13,82016.52,0,11940.39,532061.13,UPI,DTDC,TRK731489123,NaN,0,App,Above ₹5 lakh
306,ORD000307,CUST03874,20-03-2024,11:55:00,Cancelled,Raipur,Chhattisgarh,721504,550716.17,85706.88,0,40438.71,550716.17,Cash on Delivery,Delhivery,TRK499571364,NaN,1,App,Above ₹5 lakh
583,ORD000584,CUST07624,30-10-2024,02:55:00,Delivered,Hyderabad,Telangana,487237,745667.68,126685.30,0,81057.62,745667.68,Cash on Delivery,BlueDart,TRK820123719,04-11-2024,1,Website,Above ₹5 lakh
590,ORD000591,CUST05298,22-05-2025,04:28:00,Cancelled,Faridabad,Haryana,262336,520293.72,84126.96,0,31732.24,520293.72,Credit Card,Shadowfax,TRK957301189,NaN,0,App,Above ₹5 lakh
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39194,ORD039195,CUST02704,02-09-2024,10:53:00,Delivered,Guwahati,Assam,730933,588456.74,95751.18,0,41919.44,588456.74,Cash on Delivery,Amazon Logistics,TRK796217398,04-09-2024,1,Website,Above ₹5 lakh
39582,ORD039583,CUST06723,28-06-2023,07:15:00,Delivered,Meerut,Uttar Pradesh,189610,519500.64,77975.21,0,6794.57,519500.64,Debit Card,DTDC,TRK800886823,04-07-2023,0,App,Above ₹5 lakh
39793,ORD039794,CUST04753,02-02-2024,08:01:00,Delivered,Chandigarh,Chandigarh,560393,640296.82,107943.68,0,63337.86,640296.82,Credit Card,Ecom Express,TRK701697094,06-02-2024,0,App,Above ₹5 lakh
39804,ORD039805,CUST04071,11-07-2023,23:03:00,Delivered,Guwahati,Assam,714125,796481.05,120415.28,0,13403.23,796481.05,Net Banking,Ecom Express,TRK327250998,15-07-2023,0,App,Above ₹5 lakh


## Step 6: Referential Integrity Checks

In [15]:
order_items = copied_data_dict["order_items DA"]
orders = copied_data_dict["orders DA"]

invalid_order_ids = order_items[
    ~order_items["order_id"].isin(orders["order_id"])
]

print("Number of order items with invalid order_ids:", len(invalid_order_ids))

Number of order items with invalid order_ids: 0


## Step 7: Domain-Specific Operational Checks

In [16]:
orders['order_date_dt'] = pd.to_datetime(orders['order_date'], format='%d-%m-%Y', errors='coerce')
orders['delivered_date_dt'] = pd.to_datetime(orders['delivered_date'], format='%d-%m-%Y', errors='coerce')

invalid_delivery_dates = orders[orders['delivered_date_dt'] < orders['order_date_dt']]
print(f"Number of orders where delivery date precedes order date: {len(invalid_delivery_dates)}")

Number of orders where delivery date precedes order date: 0


In [17]:
payments = copied_data_dict['payments DA']
payment_status_counts = payments['status'].value_counts()
failed_count = payment_status_counts.get('Failed', 0)
total_payments = len(payments)
failure_rate = (failed_count / total_payments) * 100

print("\nPayment Status Counts:")
print(payment_status_counts)
print(f"Payment Failure Rate: {failure_rate:.2f}%")


Payment Status Counts:
status
Success    48252
Failed      1748
Name: count, dtype: int64
Payment Failure Rate: 3.50%


In [18]:
returns = copied_data_dict['returns DA']
invalid_returns = returns[~returns['order_id'].isin(orders['order_id'])]
print(f"Returns without matching order_id (Orphan returns): {len(invalid_returns)}")

Returns without matching order_id (Orphan returns): 0


In [19]:
inventory = copied_data_dict['inventory DA']
status_col = [c for c in inventory.columns if 'status' in c.lower()]
out_of_stock_count = (inventory[status_col[0]] == 'Out of Stock').sum()
print(f"Products in 'Out of Stock' status: {out_of_stock_count}")

Products in 'Out of Stock' status: 2


In [20]:
customers = copied_data_dict['customers DA']
zero_order_custs = (customers['total_orders'] == 0).sum()
print(f"Customers with total_orders '0': {zero_order_custs}")

Customers with total_orders '0': 457


In [21]:
output_dir = "indiakart_cleaned_data"
os.makedirs(output_dir, exist_ok=True)

for table_name, df in copied_data_dict.items():
    file_path = os.path.join(output_dir, f"{table_name}.csv")
    df.to_csv(file_path, index=False)
    
print(f"✓ Successfully saved all cleaned tables to ./{output_dir}/")

✓ Successfully saved all cleaned tables to ./indiakart_cleaned_data/


## Step 8 Load Cleaned Data into PostgreSQL
The final cleaned DataFrames stored in cleaned_data_dict are loaded into PostgreSQL for SQL-based analysis and KPI calculation. The cleaned data is loaded directly from the in-memory DataFrames so that cleaned data types, including datetime columns, are preserved.

In [24]:
from sqlalchemy import create_engine
from urllib.parse import quote_plus
from getpass import getpass

# PostgreSQL connection details
username = "postgres"
password = quote_plus(getpass("Enter PostgreSQL password: "))
host = "localhost"
port = "5432"
database = "indiakart"

engine = create_engine(
    f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}"
)

# Load cleaned DataFrames into PostgreSQL
for table_name, df in copied_data_dict.items():

    df.to_sql(table_name, engine, if_exists="replace", index=False)
    print(f"Loaded: {table_name}")

print("\nAll cleaned DataFrames successfully loaded into PostgreSQL.")

Enter PostgreSQL password:  ········


Loaded: customers DA
Loaded: inventory DA
Loaded: order_items DA
Loaded: orders DA
Loaded: payments DA
Loaded: products DA
Loaded: returns DA
Loaded: suppliers DA

All cleaned DataFrames successfully loaded into PostgreSQL.


## Step 9: Data Quality Summary Report

## Phase 1: Data Quality & Cleaning Report

### 1. Overview & Scope
* **Dataset Processed:** 8 tables including orders, order items, payments, returns, inventory, and customers.
* **Objective:** Profile datasets, check for missing values, identify outliers, validate data types, and ensure referential integrity.

### 2. Key Findings & Checks
* **Missing Values & Profile:** Initial data profiling completed across all tables. Null percentages and data types documented.
* **Date Validation:** Verified that `delivered_date` is always on or after `order_date` (`0` violations found).
* **Payment Transactions:** Analyzed `payments DA` table. Out of total transactions, `1,748` failed, resulting in a payment failure rate of **3.50%**.
* **Referential Integrity:** 
  * `order_items` vs `orders`: All items match valid parent order IDs (`0` orphans).
  * `returns` vs `orders`: All return records correspond to valid orders (`0` orphans).
* **Inventory & Outliers:** 
  * `inventory DA`: Exactly `2` products currently flagged as 'Out of Stock'.
  * `customers DA`: Identified `457` customer profiles with `total_orders = 0` (valid inactive profiles).

### 3. Conclusion
The dataset is structurally sound with high referential integrity. Minor data cleanliness items (such as payment failures and inactive customer profiles) have been noted for downstream analysis in Phase 2.